# Validation-only matched-window control

This experiment compares GAT-GRU and joint LSTM at the same input windows (18 and 24), using the same ten node features, five seeds, validation split, optimizer settings, and fixed training procedure. LSTM receives the same 4-dimensional service embedding and concatenates all services at each timestep; it does not receive edge features. Hidden size 73 is chosen to bring joint LSTM near the GAT-GRU parameter count (~120k; verify the recorded count).

No Optuna search or test evaluation is performed here. The result is a validation control, not a replacement final test.

In [1]:
import copy
import sys
from pathlib import Path
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / 'modeling-common').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'modeling-common'))
import forecast_ablation as fa
import forecast_torch as ft

ctx = fa.load_context(ROOT)
selection = fa.load_selection_contract(ROOT / 'configs' / 'gatgru' / 'feature_contract_v01.json')
gat_base = fa.load_baseline_config(ctx)
shared_training = {'dropout': 0.1, 'lr': 1e-3, 'batch_size': 128, 'weight_decay': 0.0}
variants = []
for window in (18, 24):
    gat_cfg = copy.deepcopy(gat_base)
    gat_cfg.update(shared_training, window=window, use_edge_features=True, emb_dim=4)
    variants.append({
        'name': f'gatgru_w{window}', 'kind': 'gatgru', 'config': gat_cfg,
        'node_features': selection['node_features'], 'edge_features': selection['edge_features'],
    })
    lstm_cfg = copy.deepcopy(ft.DEFAULT_CONFIG['lstm'])
    lstm_cfg.update(shared_training, window=window, hidden=73, layers=2, mode='joint', emb_dim=4)
    variants.append({
        'name': f'lstm_joint_h73_w{window}', 'kind': 'lstm', 'config': lstm_cfg,
        'node_features': selection['node_features'], 'edge_features': [],
    })

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = ROOT / 'artifacts' / 'comparison' / 'v02_matched_windows_validation'
summary = fa.run_fixed_variants(
    ctx, 'v02_matched_windows_validation', OUT, variants,
    seeds=(0, 1, 2, 3, 4), device=DEVICE,
    score_weights=fa.BASELINE_SCORE_WEIGHTS,
    parent_version='v05_selected_features_final',
)
display(summary)


,variant,model_kind,node_features,edge_features,edge_control,topology_control,n_seeds,primary_score_mean,primary_score_std,cpu_weighted_diagnostic_mean,...,cpu_cores_mae_macro_mean,cpu_cores_mae_macro_std,train_seconds_mean,train_seconds_std,early_stop_val_mae_mean,early_stop_val_mae_std,n_params_mean,n_params_std,epochs_trained_mean,epochs_trained_std
3,lstm_joint_h73_w24,lstm,"[rps_in, rps_in_delta, rps_per_replica, latenc...",[],real,real,5,0.112232,0.002638,0.105607,...,0.013872,0.000362,142.501943,25.339153,0.112208,0.002634,119896.0,0.0,50.0,7.000000
1,lstm_joint_h73_w18,lstm,"[rps_in, rps_in_delta, rps_per_replica, latenc...",[],real,real,5,0.113563,0.001627,0.106720,...,0.013917,0.000313,71.120489,9.485321,0.113538,0.001624,119896.0,0.0,44.8,5.805170
2,gatgru_w24,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,5,0.113720,0.000605,0.106391,...,0.013667,0.000120,1390.009263,157.403748,0.113696,0.000604,121560.0,0.0,48.4,4.979960
0,gatgru_w18,gatgru,"[rps_in, rps_in_delta, rps_per_replica, latenc...","[call_rate, call_ratio, edge_latency_p95_ms]",real,real,5,0.113944,0.000423,0.106566,...,0.013671,0.000056,839.510574,70.554259,0.113921,0.000422,121560.0,0.0,41.8,4.207137
